String Output Parser

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.6)
parser = StrOutputParser()
# 1. إنشاء القالب
template = PromptTemplate.from_template("Who is the President of country {country}")

# 2. تنسيق القالب (هذه خطوة منفصلة)
formatted_prompt = template.invoke({"country": "United States"})

# 3. إرسال للنموذج (هذه خطوة منفصلة)
result = llm.invoke(formatted_prompt)

# 4. استخراج النص (هذه خطوة منفصلة)
text_result = parser.invoke(result)

print(text_result)
print(f"نوع النتيجة: {type(text_result)}")



# parser = StrOutputParser()
# template = PromptTemplate.from_template("Who is the President of countrry {country}")

# chat = template.invoke({"country": "United States"})
# result = llm.invoke(chat)
# final_result = parser.invoke(result)
# print(final_result)

The current President of the United States is **Joe Biden**. He assumed office on January 20, 2021.
نوع النتيجة: <class 'langchain_core.messages.base.TextAccessor'>


Chain with StrOutputParser

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.6)

# 1. إنشاء القالب
template = PromptTemplate.from_template("Who is the President of country {country}")

# 2. إنشاء chain (الخطوات مرتبطة ببعض)
chain = template | llm | StrOutputParser()

# 3. خطوة واحدة فقط! 
result = chain.invoke({"country": "United States"})

print(result)
print(f"نوع النتيجة: {type(result)}")

The current President of the United States is **Joe Biden**. He is the 46th president and assumed office on January 20, 2021.
نوع النتيجة: <class 'langchain_core.messages.base.TextAccessor'>


Json Output Parser

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import json

load_dotenv()

# تهيئة النموذج
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

# إنشاء JsonOutputParser
json_parser = JsonOutputParser()

# تحسين القالب
template = PromptTemplate.from_template("""
You are a product information extractor. Extract the following product information from the text 
and return it as a valid JSON object:

Required fields:
- name: The product name (string)
- price: The product price as a number (float or integer, remove currency symbols)
- category: The product category (string)
- features: List of main features (array of strings)
- brand: The brand name (string, if available)
- availability: Is the product available? (boolean, if mentioned)

Text to analyze: 
{text}

{format_instructions}

Important: 
1. Return ONLY the JSON object, no other text
2. Ensure the JSON is valid
3. For price, extract only the number without currency symbols
4. If a field is not found, use null for that field
""",
partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

# إنشاء السلسلة
chain = template | llm | json_parser

# نص الاختبار - GAC Empow S
text = """
The GAC Empow S is a stylish sports sedan priced at $25,500. 
It belongs to the compact executive car category. 
Main features include a 1.5L turbo engine producing 177 horsepower, 7-speed DCT transmission, 
sporty exterior design with LED headlights, dual exhaust, 10.25-inch touchscreen, 
sport seats, and advanced driver assistance systems. 
The car is available in dealerships across the Middle East.
"""

# تنفيذ السلسلة
result = chain.invoke({"text": text})

print("="*50)
print("النتيجة المستخرجة - GAC Empow S")
print("="*50)
print(json.dumps(result, indent=2, ensure_ascii=False))

# التحقق من النتيجة
print("\n" + "="*50)
print("تفاصيل السيارة:")
print("="*50)
print(f"السيارة: {result.get('name', 'غير معروف')}")
print(f"السعر: ${result.get('price', 'غير معروف')}")
print(f"الفئة: {result.get('category', 'غير معروف')}")
print(f"الماركة: {result.get('brand', 'غير معروف')}")
print("المميزات:")
for i, feature in enumerate(result.get('features', []), 1):
    print(f"  {i}. {feature}")

النتيجة المستخرجة - GAC Empow S
{
  "name": "GAC Empow S",
  "price": 25500,
  "category": "compact executive car",
  "features": [
    "1.5L turbo engine producing 177 horsepower",
    "7-speed DCT transmission",
    "sporty exterior design with LED headlights",
    "dual exhaust",
    "10.25-inch touchscreen",
    "sport seats",
    "advanced driver assistance systems"
  ],
  "brand": "GAC",
  "availability": true
}

تفاصيل السيارة:
السيارة: GAC Empow S
السعر: $25500
الفئة: compact executive car
الماركة: GAC
المميزات:
  1. 1.5L turbo engine producing 177 horsepower
  2. 7-speed DCT transmission
  3. sporty exterior design with LED headlights
  4. dual exhaust
  5. 10.25-inch touchscreen
  6. sport seats
  7. advanced driver assistance systems


Pydantic Output Parser

In [9]:
from platform import release
from langchain_core.output_parsers import PydanticOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List, Optional

class Movie(BaseModel):
  titile:str = Field(..., description="The title of the movie")
  release_year: int = Field(..., description="Year the movie was released")
  genres: List[str] = Field(..., description="List of genres the movie belongs to")
  rating: float = Field(..., description="Average rating of the movie on a scale of 1 to 10")
  box_office: Optional[float] = Field(None, description="Worldwide box office in millions USD, if known")


llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.6)

pydantic_parser = PydanticOutputParser(pydantic_object=Movie)

template = PromptTemplate.from_template("""Extract movie information from the folloing text: 
{text}
{format_instructions}""",
partial_variables={"format_instructions": pydantic_parser.get_format_instructions()})

chain = template | llm | pydantic_parser

text = """
The movie 'Toy Story', directed by John Lasseter, was released in 1995.
It falls under the genres of animation, adventure, and comedy.
It holds an average rating of 8.3 out of 10.
The film grossed approximately $373.6 million at the global box office.
"""

result = chain.invoke({"text": text})
print(result)

titile='Toy Story' release_year=1995 genres=['animation', 'adventure', 'comedy'] rating=8.3 box_office=373.6
